## Check the setup and connect to the database

In [ ]:
%run "../../scripts/01-check_setup.ipynb"

## Use HANA DataFrame

A table with data already exist in your SAP HANA database, so you use [the `table()` method](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/latest/en-US/hana_ml.dataframe.html#hana_ml.dataframe.ConnectionContext.table) to instantiate a HANA DataFrame from the existing database table. 

In [ ]:
import importlib
import config_events
importlib.reload(config_events)
from config_events import HANA_TABLE_NAME, HANA_SCHEMA_NAME

In [ ]:
hdf_events=myconn.table(HANA_TABLE_NAME, schema=HANA_SCHEMA_NAME)

## Text splitting, preparing FAQs for vectorization

In [ ]:
# Applying the Text Splitter with recursive-splitting, available with hana-ml 2.23
from hana_ml.text.text_splitter import TextSplitter

splitter = TextSplitter(split_type='document', doc_type='html', keep_separator=True, overlap=4)
splitter._extend_pal_parameter({'GLOBAL_SEPARATOR':'[<h1>,<h2>,<h3>,<h4>,<h5>,<h6>]'})
hdf_events_chunked = splitter.split_text(
    hdf_events.add_id().select('ID', 'content_html'), 
    order_status=True
    )

In [ ]:
import pandas as pd
pd.set_option('max_colwidth', None) 

print(hdf_events_chunked.shape)
display(splitter.statistics_.collect())
display(hdf_events_chunked.select("*", ('LENGTH("CONTENT")', "CHUNK_SIZE")).tail(10).collect())

## Generating Text Embeddings in SAP HANA Cloud

In [ ]:
content_column = 'CONTENT'

In [ ]:
print(f"""Number of records selected for further processing: {hdf_events_chunked.count()}""")

In [ ]:
hdf_events_chunked.get_table_structure()

In [ ]:
### Generating Text Embeddings in SAP HANA Cloud with the new PAL function, function available with hana-ml 2.23.
from hana_ml.text.pal_embeddings import PALEmbeddings
pe = PALEmbeddings(model_version='SAP_GXY.20250407')
hdf_events_chunks = pe.fit_transform(hdf_events_chunked, key="SUB_ID", target=[f"{content_column}"], thread_number=10, batch_size=10) #, max_token_num=512
print(f"{hdf_events_chunks.count()} records processed in {round(pe.runtime, 3)} sec")

In [ ]:
hdf_events_chunks.get_table_structure()

In [ ]:
hdf_events_chunks.head(2).collect()

In [ ]:
print(hdf_events_chunks.select_statement)

In [ ]:
hdf_events_chunks=hdf_events_chunks.save(where="#CODECONNECT_CHUNKS_EMBEDDINGS", force=True)

## Semantic search

In [ ]:
prompt="Who will present a session about integration of Predictive Analytic Library with AI Agents? "

In [ ]:
df_result = myconn.sql(
    f"""SELECT TOP 5
    COSINE_SIMILARITY(VECTOR_EMBEDDING('{prompt}', 'QUERY', 'SAP_GXY.20250407'), "VECTOR_COL_{content_column}") AS "SIMILARITY",
    "ID", "SUB_ID", "{content_column}"
    FROM ({hdf_events_chunks.select_statement})
    ORDER BY 1 DESC;
    """
).collect()

In [ ]:
df_result.head(5)

In [ ]:
# Print the rows of the 'content' column
print(df_result['CONTENT'][0])

In [ ]:
from IPython.display import HTML

# Convert the rows of the 'content' column to markdown format
# display(HTML(df_result['CONTENT'][0]))

html_output = df_result['CONTENT'].iloc[0]

display(HTML(html_output))